In [1]:
import torch.nn as nn
import torch
import json
from PIL import Image
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from datasets import Dataset

In [2]:
torch.set_default_dtype(torch.float32)

In [3]:
class MoondreamTextWrapper(nn.Module):
    def __init__(self, text_module_dict):
        super().__init__()
        self.blocks = text_module_dict['blocks']
        self.post_ln = text_module_dict['post_ln']
        self.lm_head = text_module_dict['lm_head']
        
        # FORCE everything to float32 for CPU training
        self.blocks.to(torch.float32)
        self.post_ln.to(torch.float32)
        self.lm_head.to(torch.float32)

    def forward(self, inputs_embeds, labels=None, attention_mask=None):
        # 1. Ensure input is float32
        x = inputs_embeds.to(torch.float32)
        
        for layer_item in self.blocks:
            if isinstance(layer_item, nn.ModuleDict):
                actual_layer = list(layer_item.values())[0]
                # 2. Ensure the layer itself is float32 before calling
                x = actual_layer.to(torch.float32)(x)
            else:
                x = layer_item.to(torch.float32)(x)
            
        x = self.post_ln.to(torch.float32)(x)
        logits = self.lm_head.to(torch.float32)(x)

        loss = None
        if labels is not None:
            # CrossEntropy expects Float32 logits but Long labels
            loss_fct = nn.CrossEntropyLoss()
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            loss = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))

        return {"loss": loss, "logits": logits}

In [4]:
model_id = "vikhyatk/moondream2"

# Load tokenizer and model strictly on CPU
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    trust_remote_code=True,
    dtype=torch.float32,  # Standard CPU precision
    device_map={"": "cpu"} # Force CPU
)

# Ensure the model is explicitly on CPU
model.to("cpu")
print("Model loaded successfully on CPU in Float32.")

Model loaded successfully on CPU in Float32.


In [5]:
model.model.vision.to(torch.float32)
model.model.vision.patch_emb.to(torch.float32)

Linear(in_features=588, out_features=1152, bias=True)

In [6]:
model.model._run_vision_encoder = torch.no_grad()(
    model.model._run_vision_encoder
)

In [7]:
_original_vis_enc = model.model._vis_enc

def _vis_enc_fp32(x):
    x = x.to(torch.float32)
    return _original_vis_enc(x)

model.model._vis_enc = _vis_enc_fp32

In [8]:
trainer_model = MoondreamTextWrapper(model.model.text)

In [9]:
with open(r"D:\Y4 Research\All_PNGS_dieatary\subset1_1_merged.json", "r") as f:
    data_list = json.load(f)

# Convert your JSON into a format the Trainer understands
def process_data(item):
    return {
        "image": item['image'],
        "question": item['qa'][0]['question'],
        "answer": item['qa'][0]['answer']
    }

formatted_data = [process_data(i) for i in data_list]
dataset = Dataset.from_list(formatted_data).train_test_split(test_size=0.1)

train_dataset = dataset["train"]
eval_dataset = dataset["test"]

In [10]:
def collate_fn(batch):
    images, texts = [], []

    # ---------------------------
    # 1. Load images + build text
    # ---------------------------
    for item in batch:
        img = Image.open(item["image"]).convert("RGB")
        images.append(img)

        texts.append(
            f"Question: {item['question']}\nAnswer: {item['answer']}"
        )

    # ------------------------------------
    # 2. Image Embeddings (FORCED FLOAT32)
    # ------------------------------------
    processed_images = []

    with torch.no_grad():
        for img in images:
            # IMPORTANT: force float32 at encode time
            enc = model.encode_image(
                img,
                settings={"dtype": torch.float32}
            )

            # Handle Moondream cache structure
            if hasattr(enc, "caches") and len(enc.caches) > 0:
                tens = enc.caches[0]
                if isinstance(tens, (list, tuple)):
                    tens = tens[0]
            else:
                tens = enc

            # FINAL hard cast (safety net)
            processed_images.append(tens.to(torch.float32))

    image_embeds = torch.stack(processed_images)
    batch_size = image_embeds.shape[0]

    # Moondream outputs flattened vision tokens
    image_embeds = image_embeds.view(batch_size, -1, 2048)

    # ---------------------------
    # 3. Text Embeddings (FP32)
    # ---------------------------
    tokens_out = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True
    )

    # Text embeddings from LM head weights (float32)
    text_embeds = (
        model.model.text.lm_head.weight
        .to(torch.float32)[tokens_out["input_ids"]]
    )

    # ---------------------------
    # 4. Concatenate vision + text
    # ---------------------------
    inputs_embeds = torch.cat(
        [image_embeds, text_embeds],
        dim=1
    )

    # ---------------------------
    # 5. Labels (mask vision tokens)
    # ---------------------------
    num_image_tokens = image_embeds.shape[1]

    image_label_pad = torch.full(
        (len(batch), num_image_tokens),
        -100,
        dtype=torch.long
    )

    labels = torch.cat(
        [image_label_pad, tokens_out["input_ids"]],
        dim=1
    )

    # ---------------------------
    # 6. Attention Mask
    # ---------------------------
    attention_mask = torch.ones(
        inputs_embeds.shape[:2],
        dtype=torch.long
    )

    return {
        "inputs_embeds": inputs_embeds,
        "labels": labels,
        "attention_mask": attention_mask,
    }


In [11]:
training_args = TrainingArguments(
    output_dir=r"D:\Y4 Research\Moondreams\v1",
    use_cpu=True,
    per_device_train_batch_size=1,
    remove_unused_columns=False,  # <--- THIS IS THE KEY
    fp16=False,
    logging_steps=1,
)

In [12]:
model.to("cpu")

HfMoondream(
  (model): MoondreamModel(
    (vision): ModuleDict(
      (patch_emb): Linear(in_features=588, out_features=1152, bias=True)
      (blocks): ModuleList(
        (0-26): 27 x ModuleDict(
          (ln1): LayerNorm((1152,), eps=1e-05, elementwise_affine=True)
          (attn): ModuleDict(
            (qkv): Linear(in_features=1152, out_features=3456, bias=True)
            (proj): Linear(in_features=1152, out_features=1152, bias=True)
          )
          (ln2): LayerNorm((1152,), eps=1e-05, elementwise_affine=True)
          (mlp): ModuleDict(
            (fc1): Linear(in_features=1152, out_features=4304, bias=True)
            (fc2): Linear(in_features=4304, out_features=1152, bias=True)
          )
        )
      )
      (post_ln): LayerNorm((1152,), eps=1e-05, elementwise_affine=True)
      (proj_mlp): ModuleDict(
        (fc1): Linear(in_features=2304, out_features=8192, bias=True)
        (fc2): Linear(in_features=8192, out_features=2048, bias=True)
      )
    )
  

In [13]:
print([name for name, _ in model.named_children()])

['model']


In [14]:
print([name for name, _ in model.model.named_children()])

['vision', 'text', 'region']


In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=trainer_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=collate_fn,
)

print("Starting training on CPU...")
trainer.train()

Starting training on CPU...


Step,Training Loss
1,45.299200
2,48.926600
3,44.657600
4,50.472900
5,44.325300
6,47.741800
7,48.741100
8,40.944800
9,45.446100
10,43.839500
